# Phase 5 — FastAPI + Streamlit for Customer Support RAG



## 1. Find the project root

In [1]:
from pathlib import Path
import json, os, sys, textwrap

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "data" / "processed" / "knowledge_base_chunks.jsonl").exists()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/processed/knowledge_base_chunks.jsonl. "
        "Open this notebook from your capstone project and run Phases 1–2 first."
    )

print("Project root:", PROJECT_ROOT)
print("Phase 1 chunks:", (PROJECT_ROOT / "data/processed/knowledge_base_chunks.jsonl").exists())
print("Phase 2 FAISS index:", (PROJECT_ROOT / "data/vector_store/customer_support.faiss").exists())
assert (PROJECT_ROOT / "data/vector_store/customer_support.faiss").exists(), "Run Phase 2 first."

Project root: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents
Phase 1 chunks: True
Phase 2 FAISS index: True


## 2. Install the API and UI dependencies

In [2]:
# This installs only the service/UI packages and does not force a Transformers downgrade.
%pip install fastapi uvicorn streamlit requests httpx

  Using cached fastapi-0.143.0-py3-none-any.whl.metadata (26 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached pyarrow-25.0.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached httptools-0.9.0-cp314-cp314-win_amd64.whl.metadata (3.7 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached fastapi-0.143.0-py3-none-any.whl (144 kB)
Using cached uvicorn-0.54.0-py3-none-any.whl (87 kB)
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ------- -------------------------------- 1.8/10.3 MB 9.4 MB/s eta 0:00:01
   ---------------- ----------------------- 4.2/10.3 MB 9.6 MB/s eta 0:00:01
   ----------------------- -----

## 3. Create the project folders

In [3]:
SRC_DIR = PROJECT_ROOT / "src"
TEST_DIR = PROJECT_ROOT / "tests"
REPORT_DIR = PROJECT_ROOT / "reports"
SRC_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Created/confirmed:", SRC_DIR, TEST_DIR, REPORT_DIR)

Created/confirmed: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\src C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\tests C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\reports


## 4. Create `src/rag_service.py`

This module lazily loads the embedding model, FAISS index, BM25 index, and FLAN-T5 model when a question is asked. It uses direct `AutoModelForSeq2SeqLM` loading, which avoids the Transformers v5 `text2text-generation` pipeline error you encountered earlier.

In [4]:
rag_service_code = r"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import faiss
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

PROJECT_ROOT = Path(__file__).resolve().parents[1]
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "knowledge_base_chunks.jsonl"
FAISS_PATH = PROJECT_ROOT / "data" / "vector_store" / "customer_support.faiss"
FEEDBACK_PATH = PROJECT_ROOT / "reports" / "customer_feedback.jsonl"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL_NAME = "google/flan-t5-small"

_chunks: list[dict[str, Any]] | None = None
_embedding_model = None
_faiss_index = None
_bm25_index = None
_tokenizer = None
_generation_model = None
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", str(text).lower())


def _load_resources() -> None:
    global _chunks, _embedding_model, _faiss_index, _bm25_index
    global _tokenizer, _generation_model

    if _chunks is not None:
        return

    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(f"Knowledge-base chunks not found: {CHUNKS_PATH}")
    if not FAISS_PATH.exists():
        raise FileNotFoundError(f"FAISS index not found: {FAISS_PATH}")

    with CHUNKS_PATH.open("r", encoding="utf-8") as f:
        _chunks = [json.loads(line) for line in f if line.strip()]

    if not _chunks:
        raise ValueError("Knowledge base contains no chunks.")

    _faiss_index = faiss.read_index(str(FAISS_PATH))
    if _faiss_index.ntotal != len(_chunks):
        raise ValueError(
            f"FAISS index has {_faiss_index.ntotal} vectors but there are {len(_chunks)} chunks."
        )

    _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    texts = [str(chunk.get("text", "")) for chunk in _chunks]
    _bm25_index = BM25Okapi([_tokenize(text) for text in texts])

    _tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
    _generation_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL_NAME)
    _generation_model = _generation_model.to(_device)
    _generation_model.eval()


def health_status() -> dict[str, Any]:
    return {
        "status": "ok",
        "project": "customer-support-agentic-rag",
        "chunks_file_exists": CHUNKS_PATH.exists(),
        "faiss_index_exists": FAISS_PATH.exists(),
        "models_loaded": _chunks is not None,
    }


def _hybrid_search(query: str, top_k: int = 4, candidate_k: int = 10) -> list[dict[str, Any]]:
    _load_resources()
    assert _chunks is not None and _embedding_model is not None
    assert _faiss_index is not None and _bm25_index is not None

    n = len(_chunks)
    candidate_k = min(max(top_k, candidate_k), n)
    query_vector = _embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    semantic_scores, semantic_ids = _faiss_index.search(query_vector, candidate_k)

    token_scores = _bm25_index.get_scores(_tokenize(query))
    keyword_ids = np.argsort(token_scores)[::-1][:candidate_k]

    # Reciprocal Rank Fusion: combine rank positions instead of incomparable raw scores.
    rrf_scores: dict[int, float] = {}
    for rank, idx in enumerate(semantic_ids[0], start=1):
        if idx >= 0:
            rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0.0) + 1.0 / (60 + rank)
    for rank, idx in enumerate(keyword_ids, start=1):
        rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0.0) + 1.0 / (60 + rank)

    ranked_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    results = []
    for idx in ranked_ids:
        item = dict(_chunks[idx])
        item["rrf_score"] = float(rrf_scores[idx])
        results.append(item)
    return results


def _generate_answer(question: str, retrieved: list[dict[str, Any]]) -> str:
    assert _tokenizer is not None and _generation_model is not None
    if not retrieved:
        return "I could not find relevant information in the available support knowledge base. Please contact customer support."

    evidence_blocks = []
    used_chars = 0
    for item in retrieved:
        title = str(item.get("title", "Support document"))
        chunk_id = str(item.get("chunk_id", "unknown"))
        source = str(item.get("source", "knowledge base"))
        text = str(item.get("text", "")).strip()
        block = f"[{chunk_id}] {title} (source: {source})\n{text}"
        if used_chars + len(block) > 3000:
            break
        evidence_blocks.append(block)
        used_chars += len(block)

    prompt = (
        "Answer the customer question using only the support information below. "
        "If the information does not answer the question, say that you cannot find the answer "
        "in the available information. Do not invent policies.\n\n"
        f"SUPPORT INFORMATION:\n{chr(10).join(evidence_blocks)}\n\n"
        f"CUSTOMER QUESTION: {question}\nANSWER:"
    )
    inputs = _tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(_device)
    with torch.no_grad():
        output_ids = _generation_model.generate(
            **inputs, max_new_tokens=160, do_sample=False
        )
    answer = _tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return answer or "I could not generate a reliable answer. Please contact customer support."


def query_rag(question: str, top_k: int = 4) -> dict[str, Any]:
    question = str(question).strip()
    if not question:
        raise ValueError("Question cannot be empty.")
    retrieved = _hybrid_search(question, top_k=top_k)
    answer = _generate_answer(question, retrieved)
    sources = [
        {
            "chunk_id": str(item.get("chunk_id", "unknown")),
            "title": str(item.get("title", "Support document")),
            "source": str(item.get("source", "knowledge base")),
            "category": str(item.get("category", "general")),
        }
        for item in retrieved
    ]
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_count": len(retrieved),
    }


def save_feedback(question: str, answer: str, rating: str, comment: str = "") -> dict[str, Any]:
    FEEDBACK_PATH.parent.mkdir(parents=True, exist_ok=True)
    record = {
        "question": str(question).strip(),
        "answer": str(answer).strip(),
        "rating": str(rating).strip(),
        "comment": str(comment).strip(),
    }
    with FEEDBACK_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return {"status": "saved", "feedback_file": str(FEEDBACK_PATH)}
"""
(SRC_DIR / "rag_service.py").write_text(rag_service_code.strip() + "\n", encoding="utf-8")
print("Created:", SRC_DIR / "rag_service.py")

Created: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\src\rag_service.py


## 5. Create `src/api.py` with the three required endpoints

In [5]:
api_code = r"""
from pathlib import Path
import sys
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

SRC_DIR = Path(__file__).resolve().parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag_service import health_status, query_rag, save_feedback

app = FastAPI(
    title="Customer Support RAG API",
    description="A small API for the customer-support RAG assistant.",
    version="1.0.0",
)


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=1000)
    top_k: int = Field(default=4, ge=1, le=10)


class FeedbackRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=1000)
    answer: str = Field(..., min_length=1, max_length=5000)
    rating: Literal["positive", "negative"]
    comment: str = Field(default="", max_length=2000)


@app.get("/health")
def health():
    return health_status()


@app.post("/query")
def query(request: QueryRequest):
    try:
        return query_rag(request.question, top_k=request.top_k)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        # The API returns a clear error while debugging locally. Avoid exposing
        # detailed exception messages in a public production deployment.
        raise HTTPException(status_code=500, detail=f"Query failed: {type(exc).__name__}") from exc


@app.post("/feedback")
def feedback(request: FeedbackRequest):
    try:
        return save_feedback(
            question=request.question,
            answer=request.answer,
            rating=request.rating,
            comment=request.comment,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail="Could not save feedback.") from exc
"""
(SRC_DIR / "api.py").write_text(api_code.strip() + "\n", encoding="utf-8")
print("Created:", SRC_DIR / "api.py")

Created: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\src\api.py


## 6. Create a simple Streamlit chat interface

In [6]:
streamlit_code = r"""
import os
import requests
import streamlit as st

st.set_page_config(page_title="Customer Support Assistant", page_icon="💬", layout="centered")
st.title("Customer Support Assistant")
st.caption("Answers are based on the project's support knowledge base.")

api_url = os.getenv("RAG_API_URL", "http://127.0.0.1:8000").rstrip("/")
if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message.get("sources"):
            with st.expander("Sources"):
                for source in message["sources"]:
                    st.write(
                        f"- {source.get('title', 'Support document')} "
                        f"(ID: {source.get('chunk_id', 'unknown')}; "
                        f"source: {source.get('source', 'knowledge base')})"
                    )

question = st.chat_input("Ask a customer-support question...")
if question:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.spinner("Searching support information and preparing an answer..."):
            try:
                response = requests.post(
                    f"{api_url}/query",
                    json={"question": question, "top_k": 4},
                    timeout=180,
                )
                response.raise_for_status()
                result = response.json()
                answer = result.get("answer", "No answer was returned.")
                sources = result.get("sources", [])
                st.markdown(answer)
                if sources:
                    with st.expander("Sources"):
                        for source in sources:
                            st.write(
                                f"- {source.get('title', 'Support document')} "
                                f"(ID: {source.get('chunk_id', 'unknown')}; "
                                f"source: {source.get('source', 'knowledge base')})"
                            )
                st.session_state.messages.append(
                    {"role": "assistant", "content": answer, "sources": sources, "question": question}
                )
            except requests.RequestException as exc:
                st.error(f"Could not reach the API at {api_url}. Start the FastAPI server first. Details: {exc}")

last_answer = next(
    (m for m in reversed(st.session_state.messages) if m["role"] == "assistant" and m.get("question")),
    None,
)
if last_answer:
    st.divider()
    st.subheader("Was this answer helpful?")
    col1, col2 = st.columns(2)
    feedback_comment = st.text_input("Optional feedback comment", key="feedback_comment")
    if col1.button("👍 Helpful", use_container_width=True):
        try:
            r = requests.post(
                f"{api_url}/feedback",
                json={
                    "question": last_answer["question"],
                    "answer": last_answer["content"],
                    "rating": "positive",
                    "comment": feedback_comment,
                },
                timeout=15,
            )
            r.raise_for_status()
            st.success("Thank you! Your feedback was saved.")
        except requests.RequestException:
            st.error("Feedback could not be saved. Check that the API is running.")
    if col2.button("👎 Not helpful", use_container_width=True):
        try:
            r = requests.post(
                f"{api_url}/feedback",
                json={
                    "question": last_answer["question"],
                    "answer": last_answer["content"],
                    "rating": "negative",
                    "comment": feedback_comment,
                },
                timeout=15,
            )
            r.raise_for_status()
            st.success("Thank you. Your feedback was saved for review.")
        except requests.RequestException:
            st.error("Feedback could not be saved. Check that the API is running.")
"""
(SRC_DIR / "streamlit_app.py").write_text(streamlit_code.strip() + "\n", encoding="utf-8")
print("Created:", SRC_DIR / "streamlit_app.py")

Created: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\src\streamlit_app.py


## 7. Create basic API tests

In [7]:
test_code = r"""
from pathlib import Path
import sys

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fastapi.testclient import TestClient
from api import app

client = TestClient(app)


def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    body = response.json()
    assert body["status"] == "ok"
    assert body["chunks_file_exists"] is True
    assert body["faiss_index_exists"] is True


def test_feedback_endpoint_saves_record():
    response = client.post(
        "/feedback",
        json={
            "question": "Test question",
            "answer": "Test answer",
            "rating": "positive",
            "comment": "Automated test",
        },
    )
    assert response.status_code == 200
    assert response.json()["status"] == "saved"


def test_query_rejects_empty_question():
    response = client.post("/query", json={"question": "", "top_k": 4})
    assert response.status_code == 422
"""
(TEST_DIR / "test_api.py").write_text(test_code.strip() + "\n", encoding="utf-8")
print("Created:", TEST_DIR / "test_api.py")

Created: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\tests\test_api.py


## 8. Write a project `requirements.txt`

In [8]:
requirements_text = """fastapi
uvicorn
streamlit
requests
httpx
sentence-transformers
faiss-cpu
rank-bm25
numpy
pandas
torch
transformers
pytest
"""
(PROJECT_ROOT / "requirements.txt").write_text(requirements_text, encoding="utf-8")
print("Created:", PROJECT_ROOT / "requirements.txt")
print("Note: use the same virtual environment for the API and Streamlit UI.")

Created: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\requirements.txt
Note: use the same virtual environment for the API and Streamlit UI.


## 9. Syntax-check the generated Python files

In [9]:
import ast

files_to_check = [
    SRC_DIR / "rag_service.py",
    SRC_DIR / "api.py",
    SRC_DIR / "streamlit_app.py",
    TEST_DIR / "test_api.py",
]
for file_path in files_to_check:
    source = file_path.read_text(encoding="utf-8")
    ast.parse(source, filename=str(file_path))
    print("Syntax OK:", file_path.relative_to(PROJECT_ROOT))

print("All generated Python files passed syntax checks.")

Syntax OK: src\rag_service.py
Syntax OK: src\api.py
Syntax OK: src\streamlit_app.py
Syntax OK: tests\test_api.py
All generated Python files passed syntax checks.


## 10. Import the API and test `/health` and `/feedback`

These checks do not need to generate an answer, so they should be quick. They verify that FastAPI can import the application, that the health endpoint sees your existing Phase 1/2 artifacts, and that feedback can be saved.

In [10]:
import sys
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fastapi.testclient import TestClient
from api import app

client = TestClient(app)

health_response = client.get("/health")
print("GET /health:", health_response.status_code, health_response.json())
assert health_response.status_code == 200
assert health_response.json()["status"] == "ok"
assert health_response.json()["chunks_file_exists"] is True
assert health_response.json()["faiss_index_exists"] is True

feedback_response = client.post(
    "/feedback",
    json={
        "question": "Phase 5 test question",
        "answer": "Phase 5 test answer",
        "rating": "positive",
        "comment": "Notebook validation test",
    },
)
print("POST /feedback:", feedback_response.status_code, feedback_response.json())
assert feedback_response.status_code == 200
assert feedback_response.json()["status"] == "saved"

print("API smoke tests passed.")

c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GET /health: 200 {'status': 'ok', 'project': 'customer-support-agentic-rag', 'chunks_file_exists': True, 'faiss_index_exists': True, 'models_loaded': False}
POST /feedback: 200 {'status': 'saved', 'feedback_file': 'C:\\Users\\Tanish.Ranjan\\OneDrive - ENCORA\\Desktop\\capstone-rag-agents\\reports\\customer_feedback.jsonl'}
API smoke tests passed.


## 11. Test `/query` end-to-end

This cell loads the embedding and generation models on the first query. It may take a while because model files can be downloaded the first time. If your machine is low on memory, you can skip this cell and test later after starting the API server.

In [11]:
query_response = client.post(
    "/query",
    json={"question": "How long do I have to return an item?", "top_k": 4},
)
print("POST /query status:", query_response.status_code)

if query_response.status_code == 200:
    query_result = query_response.json()
    print("QUESTION:", query_result["question"])
    print("\nANSWER:\n", query_result["answer"])
    print("\nRETRIEVED COUNT:", query_result["retrieved_count"])
    print("\nSOURCES:")
    for source in query_result["sources"]:
        print("-", source["chunk_id"], "|", source["title"], "|", source["source"])
    assert query_result["answer"].strip(), "API returned an empty answer."
    assert isinstance(query_result["sources"], list)
    print("\nQuery endpoint test passed.")
else:
    print("API response:", query_response.text)
    raise RuntimeError(
        "The /query request failed. Review the API response above and the traceback; "
        "the health and feedback checks may still have passed."
    )

Loading weights: 100%|██████████| 190/190 [00:01<00:00, 173.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


POST /query status: 200
QUESTION: How long do I have to return an item?

ANSWER:
 30 calendar days

RETRIEVED COUNT: 4

SOURCES:
- POL002_CHUNK001 | Returns policy | sample_policy
- FAQ002_CHUNK001 | Cancel an order | sample_faq
- FAQ004_CHUNK001 | Request a refund | sample_faq
- FAQ005_CHUNK001 | Item arrived damaged | sample_faq

Query endpoint test passed.


## 12. Confirm feedback file and all generated artifacts

In [12]:
feedback_path = REPORT_DIR / "customer_feedback.jsonl"
print("API file:", (SRC_DIR / "api.py").exists())
print("RAG service file:", (SRC_DIR / "rag_service.py").exists())
print("Streamlit UI file:", (SRC_DIR / "streamlit_app.py").exists())
print("Test file:", (TEST_DIR / "test_api.py").exists())
print("Requirements file:", (PROJECT_ROOT / "requirements.txt").exists())
print("Feedback log:", feedback_path.exists())

assert all([
    (SRC_DIR / "api.py").exists(),
    (SRC_DIR / "rag_service.py").exists(),
    (SRC_DIR / "streamlit_app.py").exists(),
    (TEST_DIR / "test_api.py").exists(),
    (PROJECT_ROOT / "requirements.txt").exists(),
    feedback_path.exists(),
])
print("\nPHASE 5 API AND STREAMLIT CHECK COMPLETE.")

API file: True
RAG service file: True
Streamlit UI file: True
Test file: True
Requirements file: True
Feedback log: True

PHASE 5 API AND STREAMLIT CHECK COMPLETE.
